# Extracting new-product announcements from SEC Form 8-K filings with a local LLM

**Repository:** https://github.com/alexhubenthal/sec-8k-product-extraction

This notebook walks through the pipeline implemented in the `sec8k` package step by step on one
company, then loads the results of the full run (`output/new_products.csv`).

Pipeline: S&P 500 tickers → CIK (`company_tickers.json`) → 8-K list (browse-edgar Atom feed) →
filing documents → plain text → **LLM named-entity extraction** (Ollama + Llama 3, JSON-schema
constrained) → validation and LLM verification → CSV.

Requirements: the Ollama server running with the `llama3` model, and the environment variable
`SEC_USER_AGENT="Your Name your.email@example.com"` (SEC requires it).

In [1]:
import json
import logging
import os
import sys
from pathlib import Path

import pandas as pd

# run from the repository root so that data/ and output/ resolve
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from sec8k import edgar, llm, pipeline

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
pd.set_option("display.max_colwidth", 120)

USER_AGENT = os.environ.get("SEC_USER_AGENT", "")
assert "@" in USER_AGENT, "set SEC_USER_AGENT='Your Name your.email@example.com' before running"
client = edgar.EdgarClient(USER_AGENT, cache_dir="data/raw/http_cache")
ollama = llm.OllamaClient(model="llama3", num_ctx=8192)
ollama.check_available()
print("SEC client and Ollama ready")

SEC client and Ollama ready


## Step 1 – Ticker → CIK

`https://www.sec.gov/files/company_tickers.json` lists every registrant with a ticker. The CIK is
zero-padded to ten digits, the form EDGAR URLs expect.

In [2]:
ticker_map = edgar.load_ticker_map(client)
print(len(ticker_map), "tickers in company_tickers.json")
for symbol in ["AAPL", "LLY", "BRK.B"]:
    print(symbol, "->", edgar.lookup_cik(ticker_map, symbol))

10423 tickers in company_tickers.json
AAPL -> {'cik': '0000320193', 'ticker': 'AAPL', 'title': 'Apple Inc.'}
LLY -> {'cik': '0000059478', 'ticker': 'LLY', 'title': 'ELI LILLY & Co'}
BRK.B -> {'cik': '0001067983', 'ticker': 'BRK-B', 'title': 'BERKSHIRE HATHAWAY INC'}


## Step 2 – Recent 8-K filings of a company (Atom feed)

`https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=8-K&count={count}&output=atom`
returns the filings with their timestamp, accession number and the 8-K *items* reported. Items 2.02
(results of operations), 7.01 (Regulation FD) and 8.01 (other events) are the ones that carry press
releases, so those are the filings worth reading.

In [3]:
TICKER = "LLY"
record = edgar.lookup_cik(ticker_map, TICKER)
filings = edgar.list_8k_filings(client, record["cik"], record["ticker"], record["title"], count=10)
pd.DataFrame([f.to_dict() for f in filings])[["accession", "filing_type", "filing_date", "filed_at", "items"]]

,accession,filing_type,filing_date,filed_at,items
0,0000059478-26-000104,8-K,2026-10-29,2026-10-29T07:03:30-04:00,"[2.02, 9.01]"
1,0000059478-26-000077,8-K,2026-08-05,2026-08-05T07:05:09-04:00,"[2.02, 9.01]"
2,0001193125-26-232707,8-K,2026-05-20,2026-05-20T16:30:32-04:00,"[8.01, 9.01]"
3,0000059478-26-000048,8-K,2026-05-07,2026-05-07T16:41:48-04:00,[5.07]
4,0000059478-26-000043,8-K,2026-04-30,2026-04-30T07:03:30-04:00,"[2.02, 9.01]"
5,0000059478-26-000008,8-K,2026-02-04,2026-02-04T07:05:22-05:00,"[2.02, 9.01]"
6,0000059478-25-000271,8-K,2025-11-21,2025-11-21T16:19:05-05:00,[5.02]
7,0000059478-25-000251,8-K,2025-10-30,2025-10-30T07:03:41-04:00,"[2.02, 9.01]"
8,0001193125-25-184093,8-K,2025-08-20,2025-08-20T16:16:08-04:00,"[8.01, 9.01]"
9,0000059478-25-000202,8-K,2025-08-07,2025-08-07T07:05:40-04:00,"[2.02, 9.01]"


## Step 3 – Filing documents → plain text

The filing index page lists the documents; the primary 8-K (served through the inline-XBRL viewer,
so the `/ix?doc=` prefix is removed) and the EX-99 press-release exhibits are downloaded and converted
to text with BeautifulSoup. The text handed to the model is capped at 10,000 characters.

In [4]:
candidate = next(f for f in filings if not f.is_amendment and f.has_any_item(edgar.PRESS_RELEASE_ITEMS))
filing_text = edgar.download_filing_text(client, candidate, max_chars=10_000)
print("documents:", [(d.doc_type, d.filename) for d in filing_text.documents if d.is_primary or d.is_press_release])
print("accepted at:", filing_text.accepted_at, "| text chars:", len(filing_text.text), "| cue score:", filing_text.cue_score)
print()
print(filing_text.text[:1500], "...")

documents: [('8-K', 'lly-20261029.htm'), ('EX-99.1', 'q326lillysalesandearningsp.htm')]
accepted at: 2026-10-29 07:03:30 | text chars: 10000 | cue score: 3

FORM 8-K BODY:
UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 FORM 8-K CURRENT REPORT Pursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934 Date of Report (Date of Earliest Event Reported): October 29, 2026 ELI LILLY AND COMPANY (Exact Name of Registrant as Specified in its Charter) Indiana 001-06351 35-0470950 ( [...] Item 2.02. Results of Operations and Financial Condition. The information in this Item 2.02, including Exhibit 99.1 attached hereto, is being furnished and shall not be deemed “filed” for the purposes of Section 18 of the Securities Exchange Act of 1934 (the “Exchange Act”), or otherwise subject to the liabilities of that Section and shall not be incorporated by reference into any registration statement or other document filed pursuant to the Securities Act of 1933 or the Exc

## Step 4 – LLM named-entity extraction

The model is instructed as an NER system with two entity types, **ORGANIZATION** (the registrant)
and **PRODUCT** (a product the registrant announces as new). Ollama's structured output constrains
the answer to a JSON schema in which the verbatim `evidence` sentence comes *before* the product name,
so the entity is extracted from a quote. `postprocess()` then grounds every entity in the text
(recording its character span), checks the evidence and drops hallucinations.

In [5]:
raw, response = llm.extract_entities(
    ollama, filing_text.text, ticker=candidate.ticker, edgar_name=candidate.company,
    filing_date=candidate.filing_date, items=candidate.items,
)
print(f"LLM answered in {response.seconds:.1f}s ({response.prompt_tokens} prompt tokens, {response.completion_tokens} output tokens)")
print(json.dumps(raw, indent=2)[:2500])

LLM answered in 31.1s (3100 prompt tokens, 301 output tokens)
{
  "company_name": "Eli Lilly and Company",
  "has_new_product": true,
  "products": [
    {
      "evidence": "U.S. FDA approvals of Mounjaro to reduce cardiovascular risk in adults with type 2 diabetes;",
      "announcement_type": "regulatory_approval",
      "new_product": "Mounjaro",
      "product_description": "a treatment for reducing cardiovascular risk in adults with type 2 diabetes"
    },
    {
      "evidence": "Onswik, a once-weekly basal insulin for adults with type 2 diabetes;",
      "announcement_type": "launch",
      "new_product": "Onswik",
      "product_description": "a once-weekly basal insulin for adults with type 2 diabetes"
    },
    {
      "evidence": "Inluriyo with Verzenio for adults with ER+, HER2-, ESR1-mutated advanced or metastatic breast cancer;",
      "announcement_type": "launch",
      "new_product": "Inluriyo",
      "product_description": "a treatment for adults with ER+, HER2-, ES

In [6]:
cleaned = llm.postprocess(raw, filing_text.text, ticker=candidate.ticker,
                          fallback_company=record["title"], edgar_title=record["title"])
print("ORGANIZATION:", cleaned.company_name)
for product in cleaned.products:
    start, end = product["span"] or (0, 0)
    print(f"PRODUCT: {product['new_product']!r} [{product['announcement_type']}] span={product['span']}")
    print("   description:", product["product_description"])
    print("   in context  : ...", filing_text.text[max(0, start - 60):end + 60].replace("\n", " "), "...")
for product in cleaned.rejected:
    print("REJECTED:", product["new_product"], "->", product["reason"])

ORGANIZATION: Eli Lilly and Company
PRODUCT: 'Mounjaro' [regulatory_approval] span=(2998, 3006)
   description: A treatment for reducing cardiovascular risk in adults with type 2 diabetes
   in context  : ... 36.00. • Regulatory progress included U.S. FDA approvals of Mounjaro to reduce cardiovascular risk in adults with type 2 diabete ...
PRODUCT: 'Onswik' [launch] span=(3069, 3075)
   description: A once-weekly basal insulin for adults with type 2 diabetes
   in context  : ...  reduce cardiovascular risk in adults with type 2 diabetes; Onswik, a once-weekly basal insulin for adults with type 2 diabete ...
PRODUCT: 'Inluriyo' [launch] span=(3138, 3146)
   description: A treatment for adults with ER+, HER2-, ESR1-mutated advanced or metastatic breast cancer
   in context  : ...  once-weekly basal insulin for adults with type 2 diabetes; Inluriyo with Verzenio for adults with ER+, HER2-, ESR1-mutated adva ...
PRODUCT: 'Jaypirca' [launch] span=(3271, 3279)
   description: A treatment for

## Step 5 – Second-pass verification

Each surviving product is re-checked by a short LLM call that sees the passage around the evidence
sentence and classifies the mention into one of five categories (new launch or approval / existing
product reported / pipeline or future product / third-party product / not a product). Only the first
category is kept. A binary yes/no question proved unreliable with an 8B model, see the report.

In [7]:
for product in cleaned.products:
    context = llm.context_window(filing_text.text, product.get("evidence_span") or product.get("span"))
    verdict, reason, seconds = llm.verify_product(ollama, cleaned.company_name, product["new_product"],
                                                  product["product_description"], product["evidence"], context)
    print(f"{product['new_product']!r}: keep={verdict} ({seconds:.1f}s) - {reason}")

'Mounjaro': keep=True (5.0s) - new_launch_or_approval: The U.S. FDA approved Mounjaro to reduce cardiovascular risk in adults with type 2 diabetes.


'Onswik': keep=True (5.0s) - new_launch_or_approval: The passage mentions that Eli Lilly and Company obtained regulatory approval for Onswik, a once-weekly basal insulin for adults with type 2 diabetes.


'Inluriyo': keep=True (5.8s) - new_launch_or_approval: The passage mentions that Eli Lilly and Company received FDA approval for Inluriyo with Verzenio for adults with ER+, HER2-, ESR1-mutated advanced or metastatic breast cancer.


'Jaypirca': keep=False (5.0s) - existing_product_reported: Eli Lilly and Company obtained an expanded indication for Jaypirca in adults with untreated chronic lymphocytic leukemia or small lymphocytic lymphoma.


## Step 6 – The full run and its results

The whole pipeline (all S&P 500 companies, checkpointing, CSV outputs) is run from the command line:

```bash
python run_extraction.py            # stops after 100 product rows
```

or programmatically with `pipeline.run(pipeline.PipelineConfig(user_agent=USER_AGENT))`.
Below, the outputs of the submitted run are loaded.

In [8]:
products = pd.read_csv("output/new_products.csv")
print(products.shape[0], "product rows;", products.stock_name.nunique(), "companies; longest description:",
      products.product_description.str.len().max(), "chars")
products.head(25)

120 product rows; 48 companies; longest description: 178 chars


,company_name,stock_name,filing_time,new_product,product_description
0,AbbVie Inc.,ABBV,2026-10-30T07:51:51-04:00,Skyrizi,Subcutaneous (SC) induction in the treatment of moderately to severely active Crohn's disease (CD)
1,"Cboe Global Markets, Inc.",CBOE,2026-10-30T07:30:16-04:00,company-KPI event contracts,Event contracts for tracking key performance indicators
2,"Church & Dwight Co., Inc.",CHD,2026-10-30T07:04:05-04:00,THERABREATH™ toothpaste,A new toothpaste product line with four variants for effective cleaning and fresh flavor.
3,"Church & Dwight Co., Inc.",CHD,2026-10-30T07:04:05-04:00,DUAL DEFENSE™ with Microban® Clumping Litter,A new cat litter formula with two layers of powerful protection for odor elimination and antimicrobial properties.
4,"Church & Dwight Co., Inc.",CHD,2026-10-30T07:04:05-04:00,THERABREATH™ portable rinse sachets,Portable rinse sachets for fresh breath on-the-go.
5,"Regeneron Pharmaceuticals, Inc.",REGN,2026-10-30T06:50:20-04:00,Pasatru ™ (garetosmab),Approved by FDA for fibrodysplasia ossificans progressiva (FOP)
6,Intel Corporation,INTC,2026-10-29T16:08:53-04:00,Intel Core Ultra Series 3,Premium client platform for Googlebook devices
7,ResMed Inc.,RMD,2026-10-29T16:05:45-04:00,AirTouch F40,Fabric CPAP mask
8,"Amazon.com, Inc.",AMZN,2026-10-29T16:05:27-04:00,Bedrock Managed Agents,A managed service powered by OpenAI that makes it faster for enterprises to build and deploy production-ready AI age...
9,"Amazon.com, Inc.",AMZN,2026-10-29T16:05:27-04:00,Strands harness,"A fully assembled, state-of-the-art agent harness that developers can get working with a single line of code and bui..."


In [9]:
filings_log = pd.read_csv("output/filings_processed.csv")
summary = json.load(open("output/summary.json"))
print(json.dumps({k: summary[k] for k in ["model", "filings_processed", "filings_with_new_product", "product_rows",
                                           "companies_processed", "companies_with_product", "rejected_products",
                                           "llm_seconds_mean", "verify_seconds_mean_per_filing", "llm_seconds_total",
                                           "filing_date_min", "filing_date_max"]}, indent=2))
print("\nfilings with a product by 8-K item:", summary["items_of_filings_with_product"])
print("announcement types:", summary["announcement_types"])
print("rejection reasons:", summary["rejection_reasons"])

{
  "model": "llama3",
  "filings_processed": 213,
  "filings_with_new_product": 58,
  "product_rows": 120,
  "companies_processed": 185,
  "companies_with_product": 48,
  "rejected_products": 316,
  "llm_seconds_mean": 23.3,
  "verify_seconds_mean_per_filing": 6.1,
  "llm_seconds_total": 6243.6,
  "filing_date_min": "2026-04-30",
  "filing_date_max": "2026-10-30"
}

filings with a product by 8-K item: {'2.02': 58, '8.01': 7, '7.01': 4, '5.02': 1, '1.01': 1}
announcement types: {'launch': 117, 'unveiling': 3, 'regulatory_approval': 2}
rejection reasons: {'LLM verification': 91, 'evidence has no launch/approval/availability cue': 84, 'evidence does not mention the product': 63, 'product name not found in filing text': 25, 'initiative, partnership, agreement or opening, not a product': 17, 'generic or non-product name': 12, 'empty product name': 7, 'duplicate': 6, 'pending approval or future launch, not yet available': 5, 'evidence sentence not found in filing text': 4, 'financial instru

In [10]:
(filings_log.assign(product=filings_log.n_products > 0)
            .groupby("sector")["product"].agg(filings="size", with_product="sum")
            .sort_values("with_product", ascending=False))

,filings,with_product
sector,,
Health Care,44,18
Information Technology,46,17
Financials,29,6
Consumer Discretionary,23,5
Consumer Staples,12,5
Industrials,27,5
Communication Services,8,2
Energy,4,0
Materials,7,0


In [11]:
# the same table in the pipe-separated layout requested in the assignment
print(open("output/new_products_table.txt").read()[:3000])

Company Name | Stock Name | Filing Time | New Product | Product Description
------------------------------------------------------------------------------------------------------------------------
AbbVie Inc. | ABBV | 2026-10-30T07:51:51-04:00 | Skyrizi | Subcutaneous (SC) induction in the treatment of moderately to severely active Crohn's disease (CD)
Cboe Global Markets, Inc. | CBOE | 2026-10-30T07:30:16-04:00 | company-KPI event contracts | Event contracts for tracking key performance indicators
Church & Dwight Co., Inc. | CHD | 2026-10-30T07:04:05-04:00 | THERABREATH™ toothpaste | A new toothpaste product line with four variants for effective cleaning and fresh flavor.
Church & Dwight Co., Inc. | CHD | 2026-10-30T07:04:05-04:00 | DUAL DEFENSE™ with Microban® Clumping Litter | A new cat litter formula with two layers of powerful protection for odor elimination and antimicrobial properties.
Church & Dwight Co., Inc. | CHD | 2026-10-30T07:04:05-04:00 | THERABREATH™ portable rinse sach